# 01 – Data exploration

Overview of the raw data from the phantom scans (CT, MRI, ultrasound and X-ray).

## Setup

In [6]:
from pathlib import Path
from collections import Counter
import pydicom
import pandas as pd
import numpy as np

## File inventory

Lists all files in `data/raw` and counts them per folder and file type. Used to get an overview of how the data is organised and which formats each modality uses before loading any images.

`.DS_Store` files (macOS system files) are ignored. Files without an extension are shown as `(no ext)`, which is common for DICOM files.

In [2]:
# Path to the raw data folder
raw = Path("../data/raw") 

# List all files in the raw data folder (including subfolders)
files = []
for f in raw.rglob("*"):                        # Everything in data/raw and its subfolders (files and folders)
    if f.is_file() and f.name != ".DS_Store":   # Keep only files, skip macOS system files
        files.append(f)                         # Add the file path to the list
n_files = len(files)                            # Count the number of files found
print(f"{n_files} files\n")                   

# Count files per folder and file type
counts = Counter()                            # Create a counter to hold the counts of files per folder and type
for f in files: 
    folder = str(f.parent.relative_to(raw))   # Folder the file is in, relative to data/raw (e.g. "CT/Axial_1")
    ext = f.suffix.lower()                    # File extension in lowercase (e.g. ".dcm")
    if ext == "":                             # If the file has no extension
        ext = "(no ext)"
    counts[(folder, ext)] += 1                # Add 1 to the count for this folder and type combination

# Print the counts in a formatted table
print(f"{'Folder':<50} {'Type':<10} {'Files':>6}")
print("-" * 68)
for (folder, ext), n in sorted(counts.items()):
    print(f"{folder:<50} {ext:<10} {n:>6}")

811 files

Folder                                             Type        Files
--------------------------------------------------------------------
CT/Axial_1                                         .dcm           16
CT/Axial_2                                         .dcm           16
CT/Dose_Record                                     .dcm            1
CT/Dose_Report                                     .dcm            1
CT/Helical                                         .dcm          522
CT/Photos                                          .heic           7
MRI/3D_T1w                                         .dcm          120
MRI/3D_T2w                                         .dcm          120
US                                                 .dcm            6
US                                                 .txt            1
XRay                                               .dcm            1


## Key DICOM parameters per folder

Reads the header of one DICOM file from each folder and prints the main acquisition parameters: modality, series description, image size, pixel spacing and slice thickness. These are needed to convert pixel measurements to mm and to document the scans in Materials and methods.

In [ ]:
# Keep only DICOM files, sorted so the folders come in order
dcm_files = []
for f in files:                     
    if f.suffix.lower() == ".dcm":  # Keep only DICOM files
        dcm_files.append(f)         # Add the file path to the list
dcm_files.sort()                    # Sort the list of DICOM files

# Pick one example file per folder (the first one found)
examples = {}
for f in dcm_files:
    if f.parent not in examples:          # First file we see from this folder
        examples[f.parent] = f            # Save it as the folder's example

# Collect the key parameters for each folder in a list of rows
rows = []
for folder, f in examples.items():
    ds = pydicom.dcmread(f, stop_before_pixels=True)    # Header only (faster)
    n = sum(1 for x in dcm_files if x.parent == folder) # Number of files in this folder

    rows.append({
        "Folder": str(folder.relative_to(raw)),
        "Files": n,
        "Modality": ds.get("Modality"),
        "Series": ds.get("SeriesDescription"),
        "Size (px)": f"{ds.get('Rows')} x {ds.get('Columns')}",
        "Pixel spacing (mm)": ds.get("PixelSpacing"),
        "Slice thickness (mm)": ds.get("SliceThickness"),
    })

# Show as a table
overview = pd.DataFrame(rows)
overview = overview.replace({None: "", np.nan: ""})  # Replace None and NaN with empty string for better display
display(overview)   

,Folder,Files,Modality,Series,Size (px),Pixel spacing (mm),Slice thickness (mm)
0,CT/Axial_1,16,CT,Axial 1,512 x 512,"[0.585938, 0.585938]",1.25
1,CT/Axial_2,16,CT,Axial 2,512 x 512,"[0.585938, 0.585938]",1.25
2,CT/Dose_Record,1,SR,Dose Record,None x None,,
3,CT/Dose_Report,1,CT,Dose Report,512 x 512,,
4,CT/Helical,522,CT,Helical,512 x 512,"[0.585938, 0.585938]",0.625
5,MRI/3D_T1w,120,MR,3D T1w - 3A + 3B,512 x 512,"[0.2734, 0.2734]",1.2
6,MRI/3D_T2w,120,MR,3D T2w - 3A +3B,256 x 256,"[0.5469, 0.5469]",1.2
7,US,6,US,,768 x 1024,,
8,XRay,1,DX,PA,3032 x 2520,"[0.139, 0.139]",
